In [5]:
import pandas as pd


In [8]:
df = pd.read_csv("Quality_of_Life_final.csv")

In [9]:
print(df.columns)


Index(['Country', 'Year', 'LE_at_birth', 'Doctors_Per_10000',
       'Access_to_Electricity', 'Gender_Development_Index',
       'Gender_Inequality_Index', 'Human_Development_Index',
       'Health_Care_Index', 'Crime_Index'],
      dtype='str')


In [12]:
import pandas as pd

# Columns to check for missingness (UPDATED)
value_cols = [
    'LE_at_birth',
    'Doctors_Per_10000',
    'Access_to_Electricity',
    'Gender_Development_Index',
    'Gender_Inequality_Index',
    'Human_Development_Index',
    'Health_Care_Index',
    'Crime_Index'
]

# Create working table
country_year_missing = df[['Country', 'Year'] + value_cols].copy()

# Missing count and index
country_year_missing['Missing_Count'] = country_year_missing[value_cols].isna().sum(axis=1)
country_year_missing['Missingness_Index'] = (
    country_year_missing['Missing_Count'] / len(value_cols) * 100
).round(2)

def categorize_missing(count):
    if count == 0:
        return 'Complete'
    elif count <= 2:
        return 'Low Missingness(1-2 Varibles)'
    elif count <= 4:
        return 'Moderate Missingness(3-4 variables)'
    elif count <= 6:
        return 'High Missingness(5-6 Variables)'
    else:
        return 'Very High Missingness(6+ variables)'

country_year_missing['Missingness_Category'] = country_year_missing['Missing_Count'].apply(categorize_missing)

# Create decade-style year groups (1970–2022 dataset)
country_year_missing['Year_Group'] = (
    (country_year_missing['Year'] // 10) * 10
).astype(int).astype(str) + '-' + (
    ((country_year_missing['Year'] // 10) * 10 + 9)
).astype(int).astype(str)

# Overall missingness category summary
category_summary = (
    country_year_missing['Missingness_Category']
    .value_counts()
    .reindex(['Complete', 'Low Missingness(1-2 Varibles)', 'Moderate Missingness(3-4 variables)', 'High Missingness(5-6 Variables)', 'Very High Missingness(6+ variables)'])
    .reset_index()
)
category_summary.columns = ['Missingness_Category', 'Number_of_rows']
category_summary['Percentage (%)'] = (
    category_summary['Number_of_rows'] /
    category_summary['Number_of_rows'].sum() * 100
).round(2)

# Average missingness by year group
missing_by_year_group = (
    country_year_missing.groupby('Year_Group')['Missingness_Index']
    .mean()
    .round(2)
    .reset_index(name='Average_Missingness_Index')
)
missing_by_year_group = missing_by_year_group.sort_values(
    by='Year_Group',
    key=lambda x: x.str[:4].astype(int)
)

# Missingness category distribution within each year group
category_by_year_group = (
    country_year_missing.groupby(['Year_Group', 'Missingness_Category'])
    .size()
    .reset_index(name='Count')
)

category_by_year_group['Percentage (%)'] = (
    category_by_year_group.groupby('Year_Group')['Count']
    .transform(lambda x: (x / x.sum() * 100).round(2))
)

category_by_year_group = category_by_year_group.sort_values(
    by='Year_Group',
    key=lambda x: x.str[:4].astype(int)
)

# Print outputs
print("\n=== Overall Missingness Category Summary ===")
print(category_summary.to_string(index=False))

print("\n=== Average Missingness by Year Group ===")
print(missing_by_year_group.to_string(index=False))

print("\n=== Missingness Category by Year Group ===")
print(category_by_year_group.to_string(index=False))


=== Overall Missingness Category Summary ===
               Missingness_Category  Number_of_rows  Percentage (%)
                           Complete             563            4.14
      Low Missingness(1-2 Varibles)            2384           17.53
Moderate Missingness(3-4 variables)            2058           15.13
    High Missingness(5-6 Variables)            4107           30.20
Very High Missingness(6+ variables)            4486           32.99

=== Average Missingness by Year Group ===
Year_Group  Average_Missingness_Index
 1970-1979                      85.33
 1980-1989                      84.37
 1990-1999                      56.64
 2000-2009                      46.97
 2010-2019                      37.52
 2020-2029                      46.00

=== Missingness Category by Year Group ===
Year_Group                Missingness_Category  Count  Percentage (%)
 1970-1979     High Missingness(5-6 Variables)    440           17.36
 1970-1979 Very High Missingness(6+ variables)   2094